# Data Preparation

Load and inspect the charging session and Burbank airport weather datasets from `Data Share`. The original CSV files are not modified.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Run this notebook from the project directory.
data_dir = Path("Data Share")
if not data_dir.is_dir():
    raise FileNotFoundError("Run the notebook from the project directory containing Data Share.")

# Preserve leading zeros in identifiers and exclude the exported index column.
charging_sessions = pd.read_csv(
    data_dir / "charging_sessions.csv",
    index_col=0,
    dtype={column: "string" for column in [
        "id", "sessionID", "siteID", "spaceID", "stationID", "userID"
    ]},
)
weather = pd.read_csv(data_dir / "weather_burbank_airport.csv")

# Charging timestamps explicitly use UTC. Weather timestamps have no stated timezone.
for column in ["connectionTime", "disconnectTime", "doneChargingTime"]:
    charging_sessions[column] = pd.to_datetime(charging_sessions[column], utc=True)
weather["timestamp"] = pd.to_datetime(weather["timestamp"])

summary = pd.DataFrame({
    "Dataset": ["Charging sessions", "Burbank airport weather"],
    "Rows": [len(charging_sessions), len(weather)],
    "Columns": [charging_sessions.shape[1], weather.shape[1]],
    "Start timestamp": [str(charging_sessions["connectionTime"].min()), str(weather["timestamp"].min())],
    "End timestamp": [str(charging_sessions["connectionTime"].max()), str(weather["timestamp"].max())],
})
display(summary)

,Dataset,Rows,Columns,Start timestamp,End timestamp
0,Charging sessions,66450,12,2018-04-25 11:08:04+00:00,2021-09-14 05:43:39+00:00
1,Burbank airport weather,29244,9,2018-01-01 08:53:00,2021-01-01 07:53:00


## Charging Sessions

Each row represents a charging session. `kWhDelivered` records delivered energy. The exported CSV index is used as the DataFrame index, leaving 12 data columns.

In [2]:
display(charging_sessions.head())
display(charging_sessions[["kWhDelivered"]].describe())

,id,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,sessionID,siteID,spaceID,stationID,timezone,userID,userInputs
0,5e23b149f9af8b5fe4b973cf,2020-01-02 13:08:54+00:00,2020-01-02 19:11:15+00:00,2020-01-02 17:31:35+00:00,25.016,1_1_179_810_2020-01-02 13:08:53.870034,0001,AG-3F30,1-1-179-810,America/Los_Angeles,000000194,"[{'WhPerMile': 250, 'kWhRequested': 25.0, 'mil..."
1,5e23b149f9af8b5fe4b973d0,2020-01-02 13:36:50+00:00,2020-01-02 22:38:21+00:00,2020-01-02 20:18:05+00:00,33.097,1_1_193_825_2020-01-02 13:36:49.599853,0001,AG-1F01,1-1-193-825,America/Los_Angeles,000004275,"[{'WhPerMile': 280, 'kWhRequested': 70.0, 'mil..."
2,5e23b149f9af8b5fe4b973d1,2020-01-02 13:56:35+00:00,2020-01-03 00:39:22+00:00,2020-01-02 16:35:06+00:00,6.521,1_1_193_829_2020-01-02 13:56:35.214993,0001,AG-1F03,1-1-193-829,America/Los_Angeles,000000344,"[{'WhPerMile': 400, 'kWhRequested': 8.0, 'mile..."
3,5e23b149f9af8b5fe4b973d2,2020-01-02 13:59:58+00:00,2020-01-02 16:38:39+00:00,2020-01-02 15:18:45+00:00,2.355,1_1_193_820_2020-01-02 13:59:58.309319,0001,AG-1F04,1-1-193-820,America/Los_Angeles,000001117,"[{'WhPerMile': 400, 'kWhRequested': 8.0, 'mile..."
4,5e23b149f9af8b5fe4b973d3,2020-01-02 14:00:01+00:00,2020-01-02 22:08:40+00:00,2020-01-02 18:17:30+00:00,13.375,1_1_193_819_2020-01-02 14:00:00.779967,0001,AG-1F06,1-1-193-819,America/Los_Angeles,000000334,"[{'WhPerMile': 400, 'kWhRequested': 16.0, 'mil..."


,kWhDelivered
count,66450.000000
mean,11.787916
std,10.314789
min,0.501000
25%,5.097000
50%,9.140000
75%,14.183000
max,108.797242


## Airport Weather

Each row contains a timestamped weather observation. Measurement units are not specified in the CSV headers.

In [3]:
display(weather.head())
display(weather.describe())

,city,timestamp,temperature,cloud_cover,cloud_cover_description,pressure,windspeed,precipitation,felt_temperature
0,Burbank,2018-01-01 08:53:00,9.0,33.0,Fair,991.75,9.0,0.0,8.0
1,Burbank,2018-01-01 09:53:00,9.0,33.0,Fair,992.08,0.0,0.0,9.0
2,Burbank,2018-01-01 10:53:00,9.0,21.0,Haze,992.08,0.0,0.0,9.0
3,Burbank,2018-01-01 11:53:00,9.0,29.0,Partly Cloudy,992.08,0.0,0.0,9.0
4,Burbank,2018-01-01 12:53:00,8.0,33.0,Fair,992.08,0.0,0.0,8.0


,timestamp,temperature,cloud_cover,pressure,windspeed,precipitation,felt_temperature
count,29244,29219.000000,29224.000000,29236.000000,29158.000000,29244.000000,29218.000000
mean,2019-06-29 23:51:42.252770048,17.885622,30.107993,986.879400,8.566568,0.059360,17.731433
min,2018-01-01 08:53:00,2.000000,4.000000,971.000000,0.000000,0.000000,0.000000
25%,2018-10-06 05:38:00,13.000000,26.000000,984.500000,6.000000,0.000000,13.000000
50%,2019-06-24 00:23:00,17.000000,33.000000,986.480000,7.000000,0.000000,17.000000
75%,2020-03-25 20:08:00,22.000000,34.000000,989.110000,13.000000,0.000000,22.000000
max,2021-01-01 07:53:00,46.000000,47.000000,999.650000,57.000000,18.540000,42.000000
std,NaN,6.454604,5.400388,3.607065,6.699889,0.471682,6.321100


## Data Quality Overview

Inspect data types and missing values before cleaning or merging the datasets.

In [4]:
def column_summary(frame):
    """Summarize data types and missing values for every column."""
    return pd.DataFrame({
        "Data type": frame.dtypes.astype(str),
        "Missing values": frame.isna().sum(),
        "Missing (%)": (frame.isna().mean() * 100).round(2),
    }).rename_axis("Column")

print("Charging sessions")
display(column_summary(charging_sessions))
print("Airport weather")
display(column_summary(weather))

print("Duplicate charging session IDs:", charging_sessions["sessionID"].duplicated().sum())
print("Duplicate weather rows:", weather.duplicated().sum())
print("Charging sessions by site ID:")
display(charging_sessions["siteID"].value_counts().rename("Sessions").to_frame())

Charging sessions


,Data type,Missing values,Missing (%)
Column,,,
id,string,0,0.00
connectionTime,"datetime64[ns, UTC]",0,0.00
disconnectTime,"datetime64[ns, UTC]",0,0.00
doneChargingTime,"datetime64[ns, UTC]",4088,6.15
kWhDelivered,float64,0,0.00
sessionID,string,0,0.00
siteID,string,0,0.00
spaceID,string,0,0.00
stationID,string,0,0.00


Airport weather


,Data type,Missing values,Missing (%)
Column,,,
city,object,0,0.00
timestamp,datetime64[ns],0,0.00
temperature,float64,25,0.09
cloud_cover,float64,20,0.07
cloud_cover_description,object,20,0.07
pressure,float64,8,0.03
windspeed,float64,86,0.29
precipitation,float64,0,0.00
felt_temperature,float64,26,0.09


Duplicate charging session IDs: 1413
Duplicate weather rows: 0
Charging sessions by site ID:


,Sessions
siteID,
0001,35042
0002,29327
2,2081


## Notes for Further Preparation

- Confirm whether site IDs `0002` and `2` refer to the same site before combining them.
- Confirm the weather timestamp timezone before joining weather observations to UTC charging timestamps.
- Weather coverage ends before charging session coverage, so later charging sessions will not have matching weather observations.
- `userInputs` remains in its original text representation. Parse it separately if its nested fields are needed.
- Missing values are reported above and have not been filled or removed.